In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
import sys
!{sys.executable} -m pip install --user tensorflow

In [3]:
df = pd.read_csv("insurance.csv")

df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [4]:
df.isna().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [6]:
encoder = LabelEncoder()

df["sex"] = encoder.fit_transform(df["sex"])

df["smoker"] = encoder.fit_transform(df["smoker"])

df["region"] = encoder.fit_transform(df["region"])

In [7]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,0,27.900,0,1,3,16884.92400
1,18,1,33.770,1,0,2,1725.55230
2,28,1,33.000,3,0,2,4449.46200
3,33,1,22.705,0,0,1,21984.47061
4,32,1,28.880,0,0,1,3866.85520


In [8]:
X = df.drop("charges", axis=1)

y = df["charges"]

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [10]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

In [11]:
model = Sequential()

model.add(Dense(64, activation="relu", input_shape=(X_train.shape[1],)))

model.add(Dense(32, activation="relu"))

model.add(Dense(16, activation="relu"))

model.add(Dense(1))

C:\Users\itsme\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
model.compile(optimizer="adam",loss="mse",metrics=["mae"])


In [14]:
history = model.fit(X_train,y_train,epochs=100,batch_size=32,validation_split=0.2)

Epoch 1/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - loss: 325453312.0000 - mae: 13518.0674 - val_loss: 310332256.0000 - val_mae: 12654.2783
Epoch 2/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 325402176.0000 - mae: 13516.3037 - val_loss: 310246688.0000 - val_mae: 12651.2402
Epoch 3/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 325242816.0000 - mae: 13511.0928 - val_loss: 310000768.0000 - val_mae: 12642.8291
Epoch 4/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 324817216.0000 - mae: 13497.6123 - val_loss: 309372608.0000 - val_mae: 12622.1738
Epoch 5/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 323795296.0000 - mae: 13466.5479 - val_loss: 307988032.0000 - val_mae: 12578.1660
Epoch 6/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 321670784.0000 - mae: 13402.8672 - val_loss: 305192128.0000 - val_mae: 12492.6533
Epoch 7/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 317807840.0000 - mae: 13287.1133 - val_loss: 300252672.0000 - val_mae: 12345.9473
Epoch 8/100

In [15]:
loss, mae = model.evaluate(X_test, y_test)

print("Loss:", loss)

print("MAE:", mae)

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 28256508.0000 - mae: 3623.3435
Loss: 28256508.0
MAE: 3623.343505859375


In [16]:
predictions = model.predict(X_test)

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step


In [17]:
mse = mean_squared_error(y_test, predictions)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, predictions)

print("MSE :", mse)

print("RMSE :", rmse)

print("R2 Score :", r2)

MSE : 28256508.45838076
RMSE : 5315.685135368794
R2 Score : 0.8179920171529514


In [18]:
model.save("insurance_ann.keras")

print("Model Saved Successfully")

Model Saved Successfully


In [19]:
results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": predictions.flatten()
})

results.to_csv("prediction_results.csv", index=False)

results.head()

,Actual,Predicted
0,9095.06825,9973.721680
1,5272.17580,5888.441895
2,29330.98315,36999.394531
3,9301.89355,8135.370605
4,33750.29180,27493.750000
